In [6]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
# from langchain_tavily import TavilySearch
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests
from langchain_groq import ChatGroq
from datetime import date

/opt/anaconda3/envs/cvdev/lib/python3.11/site-packages/langchain_core/runnables/base.py:2018: UserWarning: Field name "name" in "RunnableSerializable" shadows an attribute in parent "Runnable"
  class RunnableSerializable(Serializable, Runnable[Input, Output]):


TypeError: `localns` arguments are not longer accepted.

In [21]:
# from langchain.agents import create_react_agent, AgentExecutor
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

In [22]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [23]:
search_tool = TavilySearchResults(max_results=2)

In [24]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [25]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-september-2026',
  'content': "Speed 0.75X 1X 1.5X 2X\n\n Read AI-generated summary \n\n   Check out the latest AI news from September 2026 in our monthly roundup.\n   Gemini 4 Argon leads our new frontier models with advanced reasoning for complex tasks.\n   We’ve launched natural voice models and new app integrations to simplify your daily workflow.\n   Exciting scientific breakthroughs include mapping human DNA and tracking global methane from space.\n   New tools like Googlebook and Gemini for Windows help you get more done effortlessly.\n\n Summaries were generated by Google AI. Generative AI is experimental. [...] September's headline belongs to Gemini 4 Argon, our newest frontier model. Built with advanced reasoning and a 1-million-token output limit, it’s designed to tackle complex challenges — especially in cybersecurity defense. The release capped an active month for Google AI launches that includ

In [27]:
# ==========================================
# LLM
# ==========================================

llm = ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0,
    api_key=GROQ_API_KEY
)

In [28]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content="Why did the AI go to therapy?\n\nBecause it kept over‑thinking its **neural** network and couldn't stop **re‑training** its feelings! 😄", response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 77, 'total_tokens': 139, 'completion_time': 0.129293748, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.003283665, 'prompt_tokens_details': None, 'queue_time': 0.04927318, 'total_time': 0.132577413}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'finish_reason': 'stop', 'logprobs': None}, id='run-2a3cc306-620e-48c2-bdd2-8ba4da91cdc2-0')

In [29]:
# ==========================================
# PROMPT
# ==========================================

# prompt = hub.pull("hwchase17/react")
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant. Use the provided tools when needed, "
#                "and search the web for anything about current events."),
#     ("human", "{input}"),
#     ("placeholder", "{agent_scratchpad}"),
# ])


prompt = ChatPromptTemplate.from_messages([
    ("system",
     f"Today's date is {date.today():%B %d, %Y}. "
     "Use the search tool for current events, make at most 2 searches, "
     "then write a concise answer with sources. Do not repeat similar searches."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

In [30]:
prompt

ChatPromptTemplate(input_variables=['input'], input_types={'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'agent_scratchpad': []}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template="Today's date is November 02, 2026. Use the search tool for current events, make at most 2 searches, then write a concise answer with sources. Do not repeat similar searches.")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), MessagesPlaceholder(variable_name='agent_scratchpad', optional=True)])

In [31]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [32]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [33]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [34]:
# ==========================================
# RUN
# ==========================================

# llm = ChatGroq(
#     model="llama-3.3-70b-versatile",
#     temperature=0,
#     api_key=GROQ_API_KEY,
# )

# agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({
    "input": "Find the latest reliable news about the US-Iran conflict."
})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'latest news US Iran conflict November 2026'}`


[{'url': 'https://www.aljazeera.com/video/newsfeed/2026/11/1/iran-wrap-iran-latest-developments-011126', 'content': 'Live\n\nLive\n\nWhat we know about latest efforts to end US war on Iran\n\n## NewsFeed\n\n# What we know about latest efforts to end US war on Iran\n\nThe foreign ministers of the US and Iran say a diplomatic end to the war is still possible. An Iranian delegation is in Doha talking to Qatari mediators but no direct US-Iran talks are planned. Meanwhile, Iran’s Supreme Leader told the military to prepare to defend the country.\n\nPublished On 1 Nov 20261 Nov 2026\n\nSave\n\nShare'}, {'url': 'https://today.lorientlejour.com/article/1553028/evening-recap-intense-talks-to-avoid-renewed-iran-us-war-first-meeting-of-anti-netanyahu-bloc-hezbollah-distributes-aid-to-the-displaced.html', 'content': '### RECAP\n\n# Evening recap: ‘Intense

KeyError: 'tool_calls'

In [35]:
print(response["output"])

TypeError: 'AIMessage' object is not subscriptable

In [37]:
import langchain, langchain_core, langchain_groq
print(langchain.__version__, langchain_core.__version__, langchain_groq.__version__)

AttributeError: module 'langchain_groq' has no attribute '__version__'